# Calculate error rates from Sep/17-Sep/21 in office testing. 
Testing build Survey2026 build #70 7192ca2. Built on 2026-09-16T16:13:20Z
https://github.com/ishizuki-tech/Survey2026/releases?page=1#release-build-70-7192ca2

In [1]:
import json

In [2]:
#define the path to the reference text
reference_file = "short_swahili_answers.json"
reference = reference_file

# Read from a relative path
with open(reference, 'r') as g:
    ref_data = json.load(g)

print (f"Reference data: loaded {len(ref_data)} entries from {reference_file}")
print(json.dumps(ref_data, indent=4, ensure_ascii=False))

Reference data: loaded 10 entries from short_swahili_answers.json
{
    "Q8": "Ndiyo, iliharibu majani na kupunguza mavuno.",
    "Q9": "Naweza kukubali kupoteza hadi 10% ya mavuno.",
    "Q10": "Ikiwa uharibifu unafika karibu 30%, nitabadilisha aina.",
    "Q11": "Iwe na mavuno mengi na iwe inastahimili ukame na wadudu.",
    "Q12": "Hata tani 1 kwa ekari, kama mbegu ni nzuri.",
    "Q13": "Wakati wa maua na kutengeneza punje.",
    "Q14": "Ninaweka ya chakula, nauza iliyobaki na kidogo ni ya mifugo.",
    "Q15": "Nawapa kuku na nguruwe mahindi mara mbili hadi tatu kwa wiki.",
    "Q16": "Ninanunua kwa agrovet kwa sababu ninaamini ubora wake.",
    "Q17": "Nauza kwa wafanyabiashara wa eneo kwa sababu wako karibu na hulipa haraka."
}


In [3]:
# Define the GitHub URLs of the surveys we want to test today.
# Paste the normal github.com "blob" links; they are converted to raw file URLs below
survey_urls = [
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-17/exports/survey_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_2026-09-17_16-00-23.json",
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-21/exports/survey_4f629181-d1dc-46cf-8e71-d819a4b33662_2026-09-21_14-37-06.json",
    "https://github.com/ishizuki-tech/SurveyExports/blob/main/2026-09-21/exports/survey_7db59ab6-e13d-4080-bb07-3842d8b51fe4_2026-09-21_08-45-52.json",
]

from urllib.request import urlopen

# One entry per survey: file name plus the reference/answer pairs that matched.
surveys = []
for survey_url in survey_urls:
    raw_url = survey_url.replace("https://github.com/", "https://raw.githubusercontent.com/", 1).replace("/blob/", "/", 1)
    with urlopen(raw_url) as f:
        survey_data = json.load(f)

    survey_file = raw_url.rsplit("/", 1)[-1]
    print(f"\nLoaded data from {survey_file}")

    # Free text entered for the session (e.g. who tested and when), if present.
    session_free_text = survey_data.get('meta', {}).get('session_free_text', '')
    print(f"Session free text: {session_free_text}")

    # Keep only the questions that have a reference answer, in reference order.
    questions = [q for q in ref_data if q in survey_data['answers']]
    missing = [q for q in ref_data if q not in survey_data['answers']]
    if missing:
        print(f"Not in survey, skipped: {missing}")

    references = [ref_data[q] for q in questions]
    answers = [survey_data['answers'][q]['answer'] for q in questions]
    print(f"Found {len(answers)} transcribed answers matching the reference.")
    for q, ans in zip(questions, answers):
        print(f"{q}: {ans}")

    surveys.append({"file": survey_file, "references": references, "answers": answers})


Loaded data from survey_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_2026-09-17_16-00-23.json
Session free text: This is Victor Kitoto filling in the survey on 17th September 2026
Found 10 transcribed answers matching the reference.
Q8: Ndiyo, elia ribu majani na kupunguza mawuno.
Q9: Na eza kukubali kupoteza hadia silimi ya kumi ya mawono.
Q10: Ikiva huwari defu inafika karibu asile miatela tini, intaba deleisha maa aina.
Q11: Iwe na mawuna mengi na iwe inasta emili ukame na wadulu
Q12: Atata ni moja kwaekari kama mbegunin zori.
Q13: Wakati wamaoa na kutengeneza punje
Q14: Ninaweka ya chakula na uza iliobaki na kidobu ni amifugo.
Q15: Nwaapa koko nanguruwe maindi marambili hadi tatu kua wiki.
 Nwaapa koko nanguruwe maindi marambili hadi tatu kua wiki.
Q16: Nina nunua kwa agrovet kwa sababu nina aminiu borawake.
Q17: Na uza kwa fanya bia shara wa ineu kwa sababu wa kukaribu na uli paraka.

Loaded data from survey_4f629181-d1dc-46cf-8e71-d819a4b33662_2026-09-21_14-37-06.json
Session free text:

In [4]:
import jiwer 

transform_words = jiwer.Compose ([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(), 
    jiwer.ReduceToListOfListOfWords()
])

transform_chars = jiwer.Compose ([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(), 
    jiwer.ReduceToListOfListOfChars()
])

def error_rates(references, answers):
    out = jiwer.process_words(references, answers, reference_transform=transform_words, hypothesis_transform=transform_words)
    out2 = jiwer.process_characters(references, answers, reference_transform=transform_chars, hypothesis_transform=transform_chars)
    return out.wer, out2.cer

# Per-survey rates.
for s in surveys:
    wer, cer = error_rates(s["references"], s["answers"])
    print(f"{s['file']}  ({len(s['answers'])} answers)")
    print(f"  WER (normalized): {wer:.3f}   CER (normalized): {cer:.3f}")

# Aggregate over every answer from every survey: total errors / total reference words (or characters).
all_references = [r for s in surveys for r in s["references"]]
all_answers = [a for s in surveys for a in s["answers"]]
wer, cer = error_rates(all_references, all_answers)
print(f"\nAggregate over {len(surveys)} surveys, {len(all_answers)} answers")
print(f"  WER (normalized): {wer:.3f}   CER (normalized): {cer:.3f}")

survey_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_2026-09-17_16-00-23.json  (10 answers)
  WER (normalized): 0.814   CER (normalized): 0.313
survey_4f629181-d1dc-46cf-8e71-d819a4b33662_2026-09-21_14-37-06.json  (10 answers)
  WER (normalized): 0.698   CER (normalized): 0.178
survey_7db59ab6-e13d-4080-bb07-3842d8b51fe4_2026-09-21_08-45-52.json  (10 answers)
  WER (normalized): 0.744   CER (normalized): 0.178

Aggregate over 3 surveys, 30 answers
  WER (normalized): 0.752   CER (normalized): 0.223


## Extra speech in the transcriptions

Surveys from Sep/17–Sep/21: one of the 30 scored answers contains extra text. The other 29 start and end where the reference does; their errors are misheard or split words, not extra speech.

| Survey | Question | Extra text |
|---|---|---|
| 2026-09-17 (`46ec4547`, Victor Kitoto) | Q15 | The whole answer appears twice, on two lines: "Nwaapa koko nanguruwe maindi marambili hadi tatu kua wiki." |

The two lines are identical, so this is a repeat rather than someone else speaking. Either the respondent said the answer twice, or the app kept two transcriptions of the same recording. Listen to the audio to tell which.

No other voices, interviewer remarks or stray words (like the "Okay." or "na una prashto wa?" seen in the Sep/3–Sep/8 surveys) appear in these three surveys.

### Effect on the error rates

Scored with the repeated line removed (the calculation above keeps it):

| | WER | CER |
|---|---|---|
| 2026-09-17 as recorded | 0.814 | 0.313 |
| 2026-09-17 without the repeat | 0.733 | 0.207 |
| **Aggregate as recorded** | **0.752** | **0.223** |
| **Aggregate without the repeat** | **0.725** | **0.188** |